In [1]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

In [2]:
from rag_modules.data_preparation import DataPreparationModule
data_module = DataPreparationModule('../../data/C8/cook')

In [3]:
# 1. 准备菜谱文档
docs = data_module.load_documents()[: 100]

In [10]:
# 2. 初始化【关键词检索器】 (BM25 - Sparse)
# BM25 直接基于分词和词频计算，不需要大模型的 Embedding
bm25_retriever = BM25Retriever.from_documents(docs, k=5)  # 设置 BM25 召回 2 条记录

In [11]:
# 3. 初始化【向量检索器】 (Chroma - Dense)
embeddings = HuggingFaceEmbeddings(model='BAAI/bge-small-zh-v1.5')
vectorstore = Chroma.from_documents(docs, embeddings)
vectorstore_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# 4. 构建【混合检索器】 (EnsembleRetriever)
# weights 参数控制两者的权重比例，这里设置为五五开
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vectorstore_retriever],
    weights=[0.3, 0.7]
)

In [9]:
# 5. 测试混合检索效果
query = "番茄炒蛋怎么做？" 
# query = "水煮鱼怎么做？"
print(f"正在混合检索: '{query}'\n" + "-"*30)

results = hybrid_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"第 {i+1} 名: {doc.page_content}")

正在混合检索: '番茄炒蛋怎么做？'
------------------------------
第 1 名: # 西红柿炒鸡蛋的做法

西红柿炒蛋是中国家常几乎最常见的一道菜肴。它的原材料易于搜集，制作步骤也较为简单，所以非常适合新厨师上手，是很多人学习做菜时做的第一道菜。

预估烹饪难度：★★

## 必备原料和工具

* 西红柿
* 鸡蛋
* 食用油
* 盐
* 糖（可选）
* 葱花（可选）

## 计算

每次制作前需要确定计划做几份。一份正好够 1 个人食用

总量：

* 西红柿 = 1 个（约 180g） * 份数
* 鸡蛋 = 1.5 个 * 份数，向上取整
* 食用油 = 4ml * 鸡蛋/个
* 盐 = 1.5-2g * 份数
* 糖 = 0-2g * 份数
* 葱花 = 0-10g * 份数

## 操作

- 西红柿洗净
- 可选：去掉西红柿的外表皮
  - 开水烫表皮，然后将西红柿放入冷水，剥去外皮
- 西红柿去蒂，切成边长不超过 4cm 的小块，即为 `西红柿块`
- 将鸡蛋打入碗中，加入 `1g * 份数` 的盐，搅匀，即为 `鸡蛋液`
  - 可以考虑向鸡蛋中加入 1ml 醋，这可以去除腥味，令鸡蛋更蓬松
- 热锅，加入食用油
- 油热后，倒入 `鸡蛋液`。翻炒至鸡蛋结为固体且颜色微微发黄，即为 `半熟鸡蛋`
- 关火。将 `半熟鸡蛋` 盛盘，重新开火
  - 注意：不要洗锅
- 加入 `西红柿块`，锅铲拍打并翻炒 20 秒，或至西红柿软烂
- 向锅中加入 `半熟鸡蛋`，翻炒均匀
  - 可以考虑加入 10ml 番茄酱和 50ml 清水，增加汤汁
  - 可以额外加入一些其它熟肉和材料
- 加入剩余的盐、糖（可选，如果倾向于甜味版本）、葱花（可选），翻炒均匀
- 关火，盛盘

## 附加内容

这道菜根据不同的口味偏好，存在诸多版本，包括且不限于：

* 快速做法：
  - 直接在有`半熟鸡蛋`的锅中加入 `西红柿块`，与鸡蛋一起翻炒直至西红柿软烂
  - 继续 第 10 步
* 加入盐 改为 “加入两滴生抽”
第 2 名: # 美式炒蛋的做法

美式炒蛋具有松软鲜嫩的口感,与平时的炒蛋不同,美式炒蛋中加入了少量牛奶,使得蛋花更加的细密均匀,并且营养丰富~

预估烹饪难度：★★

## 必备原料和工具

- 鸡蛋
- 全脂牛奶/奶油

In [10]:
from rag_modules import DataPreparationModule
def extract_filters_from_query(query):
    """
    从用户问题中提取元数据过滤条件
    """
    filters = {}
    # 分类关键词
    category_keywords = DataPreparationModule.get_supported_categories()
    for cat in category_keywords:
        if cat in query:
            filters['category'] = cat
            break

    # 难度关键词
    difficulty_keywords = DataPreparationModule.get_supported_difficulties()
    for diff in sorted(difficulty_keywords, key=len, reverse=True):
        if diff in query:
            filters['difficulty'] = diff
            break

    return filters

In [11]:
extract_filters_from_query(query)

{}